<div style="font-family: system-ui, -apple-system, sans-serif; text-align: center; padding: 48px 24px 24px;">
    <div style="display: inline-block; background: #be0f05; color: white;
                padding: 12px 20px; border-radius: 12px; margin-bottom: 20px;">
        <span style="font-size: 36px; font-weight: 800; letter-spacing: -0.5px;">TIP4PATLIBS &ndash; Applicant Consolidation</span>
    </div>
    <div style="font-size: 16px; color: #475569; margin-bottom: 8px; line-height: 1.6;">
        PATSTAT knows no companies, only names. Four queries to find
        <strong>all name variants of one organisation</strong> &mdash; and count its portfolio without double-counting.
    </div>
    <div style="font-size: 13px; color: #94a3b8; margin-bottom: 32px;">
        EPO Academy Training Material &nbsp;&middot;&nbsp; <a href="https://patentreports.depa.tech" target="_blank"
           style="color: #be0f05; text-decoration: none; font-weight: 600;">created by Arne Kr&uuml;ger</a>
        &nbsp;&middot;&nbsp; inspired by Riccardo Priore
    </div>
    <div style="background: #f8fafc; border-radius: 12px; padding: 24px 28px; max-width: 640px;
                margin: 0 auto; border: 1px solid #e2e8f0; text-align: left;">
        <div style="font-size: 14px; color: #334155; line-height: 1.9;">
            <strong>What you will do in this notebook</strong>
            <br/>Step&nbsp;1 &nbsp;&middot;&nbsp; <strong>Find the names</strong> &mdash; one company, many spellings
            <br/>Step&nbsp;2 &nbsp;&middot;&nbsp; <strong>Decide</strong> what belongs together &mdash; the one step no query can do
            <br/>Step&nbsp;3 &nbsp;&middot;&nbsp; <strong>Count the group</strong> &mdash; without counting anything twice
            <br/>Step&nbsp;4 &nbsp;&middot;&nbsp; <strong>Profile the group</strong> &mdash; where it files, what it works on
            <br/>Step&nbsp;5 &nbsp;&middot;&nbsp; <strong>From notebook to application</strong>
        </div>
    </div>
    <div style="background: #fdf2f2; border-radius: 10px; padding: 16px 24px; max-width: 640px;
                margin: 28px auto 0; border: 1px solid #fecaca;">
        <div style="font-size: 14px; color: #404955; font-weight: 600;">&#9654; &nbsp;Run the cells top to bottom &mdash; about 20 seconds in total.</div>
        <div style="font-size: 12px; color: #64748b; margin-top: 6px; line-height: 1.6;">
            The default example is <strong>Siemens Healthineers</strong> and runs out of the box.
            To profile a company of your own, change <code>SEARCH_TERM</code> in the setup cell.
            Every query runs directly on PATSTAT inside EPO&nbsp;TIP &mdash; no BigQuery, no extra credentials.
            <br/>You do <strong>not</strong> need to read the SQL to follow along. Each step says in one line what its
            query asks; the SQL is there so you can see that nothing is hidden.
        </div>
    </div>
    <div style="margin-top: 20px; font-size: 12px; color: #cbd5e1;">
        Part of EPO TIP Working Group Sessions, 2026. &nbsp;Data: EPO PATSTAT Global, Autumn 2025.
    </div>
</div>

---

## Why this notebook

A client asks how big a company's patent portfolio is. You run a name search and get two hundred
rows. **None of them is the answer.** PATSTAT does not know companies — it knows names, as typed
on each application: one entry per spelling, per country, per subsidiary, per typo. Take the
biggest single row and you have undercounted the group, as a clean and confident-looking number.

Deciding which names belong together is a judgement, not a lookup, and it is the one step no
query makes for you. The assistant writes every query below; the decision is yours — and so is
writing down what you decided.

**At the end you have** one consolidated applicant group, its family count — below the naive sum
— and a written note of what your consolidation missed.


## Setup

Connect to PATSTAT and set the two parameters this notebook runs on: **which company**
to look for, and **which years** to count. The same time window is used by every query
below, so the numbers stay comparable.

`run_query` sends SQL to PATSTAT and hands back a table, printing how long it took.

In [ ]:
from epo.tipdata.patstat import PatstatClient
import pandas as pd
import time

# ── Your parameters ───────────────────────────────────────────────
SEARCH_TERM = "Siemens Healthineers"   # ← the company you are looking for
YEAR_FROM, YEAR_TO = 2014, 2024        # ← the time window for every query
# ──────────────────────────────────────────────────────────────────

# Connect to PATSTAT (PROD = the full production database on TIP)
patstat = PatstatClient(env='PROD')

def run_query(query):
    """Execute SQL on PATSTAT and return a DataFrame with timing info."""
    start = time.time()
    res = patstat.sql_query(query, use_legacy_sql=False)
    elapsed = time.time() - start
    df = pd.DataFrame(res)
    print(f"Query took {elapsed:.2f}s ({len(df)} rows)")
    return df

print("Connected to PATSTAT. Ready to run queries.")

## Step 1 &middot; Find the names

PATSTAT does not know companies. It knows **names**, exactly as they were typed on each
application &mdash; one entry per spelling, per country, per subsidiary, per typo. There is
no "Siemens Healthineers" record to look up.

So the first query asks the only question PATSTAT can answer:
**which names start with our search term, and how big is each one?**

In [ ]:
df_variants = run_query(f"""
SELECT p.person_name  AS name,
       p.person_ctry_code AS country,
       COUNT(DISTINCT a.docdb_family_id) AS families
FROM tls206_person p
JOIN tls207_pers_appln pa ON p.person_id = pa.person_id
JOIN tls201_appln     a  ON pa.appln_id  = a.appln_id
WHERE pa.applt_seq_nr > 0                        -- applicant, not inventor
  AND UPPER(p.person_name) LIKE '{SEARCH_TERM.upper()}%'
  AND a.appln_filing_year BETWEEN {YEAR_FROM} AND {YEAR_TO}
GROUP BY p.person_name, p.person_ctry_code
ORDER BY families DESC
LIMIT 200
""")
df_variants

Two things worth knowing about this hit list:

- It is a **prefix search**. `Healthineers Siemens` would not be found. Pick a term that
  sits at the *beginning* of the name.
- Anyone who now takes the biggest single row and calls it "the portfolio" has just
  undercounted the company by a wide margin. That is the whole problem in one screen.

## Step 2 &middot; Decide what belongs together

This is the step **no query can do for you**, and it is the reason this method needs a
human: are these names the same organisation?

Here the answer is easy &mdash; every hit is a Siemens Healthineers entity, so we keep all
of them. Had we searched for `Siemens` instead, the list would also contain Siemens
Energy and Siemens Mobility, and we would have to drop them by hand.

Your decision becomes one line of code &mdash; the list of names the next queries will treat
as a single company:

In [ ]:
# Names that do NOT belong to this company — add them here to exclude them.
DROP = []

group = list(dict.fromkeys(n for n in df_variants["name"] if n not in DROP))
name_filter = "p.person_name IN (" + ", ".join(
    "'" + n.replace("'", "''") + "'" for n in group) + ")"

naive_sum = df_variants.loc[~df_variants["name"].isin(DROP), "families"].sum()
print(f"{len(group)} names grouped into one company")
print(f"Naive sum of their family counts: {naive_sum}")

## Step 3 &middot; Count the group &mdash; without counting anything twice

Now the same company is filed under many names, and many of those filings protect **the
same invention**. If we simply added the numbers from Step 1 up, every invention filed
under two spellings would count twice.

PATSTAT already solves this: filings that protect one invention share a **patent family**
id. Counting families instead of applications is the entire trick &mdash; and it is why the
number below comes out *lower* than the naive sum you just printed.

In [ ]:
df_trend = run_query(f"""
SELECT a.appln_filing_year AS year,
       COUNT(DISTINCT a.docdb_family_id) AS families
FROM tls201_appln         a
JOIN tls207_pers_appln pa ON a.appln_id  = pa.appln_id
JOIN tls206_person     p  ON pa.person_id = p.person_id
WHERE {name_filter}
  AND pa.applt_seq_nr > 0
  AND a.appln_filing_year BETWEEN {YEAR_FROM} AND {YEAR_TO}
GROUP BY a.appln_filing_year
ORDER BY year
""")

print(f"Consolidated: {df_trend['families'].sum()} families  "
      f"(naive sum was {naive_sum} — the difference is the double counting we just removed)")
df_trend

**Your sanity check:** the consolidated number must always come out **at or below**
the naive sum. If the two are equal, no invention was filed under more than one spelling
&mdash; unusual for a real corporate group, and a hint to go back to Step 2.

## Step 4 &middot; Profile the group

The consolidated list of names is now just a filter you can reuse. Any question you could
ask about one applicant, you can now ask about the whole group &mdash; the decision from
Step 2 carries through unchanged.

Two examples: **where does it file?** and **what does it work on?**

In [ ]:
# Where does the group file? — grouped by the office an application was filed at
df_auth = run_query(f"""
SELECT a.appln_auth AS authority,
       COUNT(DISTINCT a.docdb_family_id) AS families
FROM tls201_appln         a
JOIN tls207_pers_appln pa ON a.appln_id  = pa.appln_id
JOIN tls206_person     p  ON pa.person_id = p.person_id
WHERE {name_filter}
  AND pa.applt_seq_nr > 0
  AND a.appln_filing_year BETWEEN {YEAR_FROM} AND {YEAR_TO}
GROUP BY a.appln_auth
ORDER BY families DESC
LIMIT 15
""")
df_auth

In [ ]:
# What does the group work on? — CPC classification, cut to subclass level (e.g. A61B)
df_cpc = run_query(f"""
SELECT SUBSTR(c.cpc_class_symbol, 1, 4) AS cpc,
       COUNT(DISTINCT a.docdb_family_id) AS families
FROM tls201_appln         a
JOIN tls207_pers_appln pa ON a.appln_id  = pa.appln_id
JOIN tls206_person     p  ON pa.person_id = p.person_id
JOIN tls224_appln_cpc  c  ON a.appln_id  = c.appln_id
WHERE {name_filter}
  AND pa.applt_seq_nr > 0
  AND a.appln_filing_year BETWEEN {YEAR_FROM} AND {YEAR_TO}
GROUP BY cpc
ORDER BY families DESC
LIMIT 15
""")
df_cpc

## Step 5 &middot; From notebook to application

Four queries and one human decision &mdash; that is applicant consolidation. Everything you
just ran by hand is exactly what the **PATSTAT Explorer** does behind its "Applicant
Search": Step&nbsp;1 fills the hit list, Step&nbsp;2 becomes checkboxes, Steps&nbsp;3 and
4 become the charts.

<div style="background: #fdf2f2; border-radius: 10px; padding: 16px 20px; margin: 20px 0;
            border: 1px solid #fecaca;">
<strong>This is the point worth taking home.</strong> On EPO&nbsp;TIP you can build a real
application on top of PATSTAT &mdash; and still see, at any moment, <em>which query</em>
produced the number on screen. The app sends the same SQL to the same database
(<code>PatstatClient(env="PROD")</code>); nothing is precomputed, nothing is hidden.
An analysis you cannot open up is one you cannot defend.
</div>

Meet the application in **`2_PATSTAT_Explorer_application.ipynb`**, and see the same
family-counting logic applied to a whole region instead of a single company in
**`4_lead_generation/1_regional-leads.ipynb`**.

### What this can and cannot see

| | |
|---|---|
| **Missed** | A subsidiary whose name does not start with your search term &mdash; the prefix search never sees it. Run a second search term for it. |
| **Over-collected** | A broad term like `Siemens` pulls in unrelated divisions. That is what Step&nbsp;2 is for. |
| **Capped** | The hit list stops at 200 names. Very broad terms lose the long tail. |
| **Judgement call** | Whether a subsidiary counts as part of the group is a question about the company, not about the data. Decide it deliberately &mdash; and write down what you decided. |